# NeuroScreen — Phase 3: SSBD Behavior Model (No Google Drive version)

**Your job (Phase 3):** Video → MediaPipe Pose → 33 Pose Landmarks → LSTM → Behavior Class
**Classes:** armFlapping, headBanging, spinning
**Output file you must produce:** `behavior_model.pth`

**Why this version is different:** Google Drive mounting kept failing (`credential propagation was unsuccessful`), so this version doesn't use Drive at all. Instead:
- Everything saves to Colab's own temporary storage.
- Every 5 epochs, the checkpoint file **automatically downloads to your computer's Downloads folder** — that's your safeguard against losing progress.
- If Colab disconnects mid-training, you re-upload your last downloaded checkpoint (one cell for that below) and training picks up from there.

**How to use this notebook:**
1. Open Google Colab → File → Upload notebook → pick this `.ipynb` file.
2. Runtime → Change runtime type → GPU (not required, but faster).
3. Run each cell top to bottom, in order. Don't skip cells.
4. Cells marked **ACTION NEEDED** need you to do something manually (upload a file, or allow a download).

**Time estimate:** ~20–45 minutes for pose extraction (the slow part), then a few minutes to actually train 20 epochs. Budget about an hour, start today.

## Step 1 — Set up local folders (no Drive needed)

In [ ]:
import os

SSBD_DIR = '/content/NeuroScreen/SSBD'
os.makedirs(SSBD_DIR, exist_ok=True)
print("Saving everything for this phase to:", SSBD_DIR)
print("Note: this folder lives only in this Colab session. Downloaded checkpoints on your computer are your real backup.")

Saving everything for this phase to: /content/NeuroScreen/SSBD
Note: this folder lives only in this Colab session. Downloaded checkpoints on your computer are your real backup.


## Step 2 — Install what we need

In [ ]:
!pip install kaggle mediapipe opencv-python-headless tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.2 MB/s eta 0:00:00


## Step 3 — Connect to Kaggle (ACTION NEEDED)

You need a Kaggle API key to download the dataset from code. This step does NOT use Google Drive, so it should work fine.

1. Go to https://www.kaggle.com/settings (sign in first).
2. Scroll to **API** → click **Create New Token**. This downloads a file called `kaggle.json`.
3. Run the cell below, then click **Choose Files** and select that `kaggle.json`.

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('KGAT_8be08ccd81543c638e91166137c71be6')
os.chmod('/root/.kaggle/access_token', 0o600)

print("Kaggle token saved.")

Kaggle token saved.


## Step 4 — Download the SSBD dataset (your Phase 3 dataset)

In [ ]:
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d shradheypathak/ssbd3 -p {DATA_DIR} --unzip

for root, dirs, filenames in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for f in filenames[:5]:
            print(f'{indent}  {f}')
        if len(filenames) > 5:
            print(f'{indent}  ... ({len(filenames)} files total)')

Dataset URL: https://www.kaggle.com/datasets/shradheypathak/ssbd3
License(s): unknown
100% 1.88G/1.88G [00:25<00:00, 79.4MB/s]

data/
  ssbd2/
    armFlapping/
      armFlapping/
    spinning/
      spinning/
    headBanging/
      headBanging/


## Step 5 — Find the video files

Searches recursively so we don't have to guess the exact nested path. **Check the counts look reasonable (not zero) before moving on.**

In [ ]:
import glob

CLASSES = ['armFlapping', 'headBanging', 'spinning']
VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

class_videos = {}
for cls in CLASSES:
    found = []
    for ext in VIDEO_EXTS:
        found.extend(glob.glob(os.path.join(DATA_DIR, '**', cls, '**', f'*{ext}'), recursive=True))
    class_videos[cls] = sorted(set(found))
    print(f"{cls}: {len(class_videos[cls])} video files found")

total = sum(len(v) for v in class_videos.values())
print(f"\nTotal: {total} videos")
assert total > 0, "No videos found - check the folder structure printed in Step 4 and adjust CLASSES/paths above."


armFlapping: 57 video files found
headBanging: 63 video files found
spinning: 63 video files found

Total: 183 videos


## Step 6 — Extract pose landmarks from each video

30 evenly-spaced frames per video, run MediaPipe Pose on each, record 33 landmarks (x, y, z, visibility = 132 numbers per frame). Turns every video into a fixed-size sequence of shape `(30, 132)`.

This is the slow step — it processes every sampled frame of every video.

In [ ]:
import urllib.request
model_path = '/content/pose_landmarker.task'
urllib.request.urlretrieve(
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task',
    model_path
)

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm.notebook import tqdm

NUM_FRAMES = 30
NUM_LANDMARKS = 33
FEATURES_PER_FRAME = NUM_LANDMARKS * 4

base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(base_options=base_options, running_mode=vision.RunningMode.IMAGE)
landmarker = vision.PoseLandmarker.create_from_options(options)

def extract_landmarks_from_video(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None
    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames).astype(int)
    sequence = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            sequence.append(np.zeros(FEATURES_PER_FRAME, dtype=np.float32))
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        if result.pose_landmarks:
            row = []
            for lm in result.pose_landmarks[0]:
                row.extend([lm.x, lm.y, lm.z, lm.visibility])
            sequence.append(np.array(row, dtype=np.float32))
        else:
            sequence.append(np.zeros(FEATURES_PER_FRAME, dtype=np.float32))
    cap.release()
    return np.stack(sequence)

## Step 7 — Build the dataset (cached locally + auto-downloaded as backup)

Feature extraction is slow, so once it's done we cache it as a `.npy` file. Since we're not using Drive, this cell also triggers a download of that cache file to your computer — if the session disconnects later, you can re-upload it instead of re-extracting from scratch (see Step 7b).

In [ ]:
CACHE_X = os.path.join(SSBD_DIR, 'X_features.npy')
CACHE_Y = os.path.join(SSBD_DIR, 'y_labels.npy')

if os.path.exists(CACHE_X) and os.path.exists(CACHE_Y):
    print("Found local cached features, loading those instead of re-extracting.")
    X = np.load(CACHE_X)
    y = np.load(CACHE_Y)
else:
    X_list, y_list = [], []
    label_map = {cls: i for i, cls in enumerate(CLASSES)}

    for cls in CLASSES:
        for video_path in tqdm(class_videos[cls], desc=cls):
            seq = extract_landmarks_from_video(video_path)
            if seq is not None:
                X_list.append(seq)
                y_list.append(label_map[cls])

    X = np.stack(X_list)
    y = np.array(y_list, dtype=np.int64)

    np.save(CACHE_X, X)
    np.save(CACHE_Y, y)
    print("Saved extracted features locally.")

    from google.colab import files
    files.download(CACHE_X)
    files.download(CACHE_Y)
    print("Downloaded X_features.npy and y_labels.npy to your computer as backup.")

print("X shape:", X.shape, " y shape:", y.shape)

armFlapping:   0%|          | 0/57 [00:00<?, ?it/s]

headBanging:   0%|          | 0/63 [00:00<?, ?it/s]

spinning:   0%|          | 0/63 [00:00<?, ?it/s]

Saved extracted features locally.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded X_features.npy and y_labels.npy to your computer as backup.
X shape: (183, 30, 132)  y shape: (183,)


### Step 7b — Only run this if you're resuming after a disconnect (ACTION NEEDED)

Skip this cell on a first run. If your session disconnected and you already have `X_features.npy` and `y_labels.npy` downloaded from before, run this to re-upload them instead of re-extracting from every video again.

In [ ]:
RESUME_FEATURES = False  # change to True if you're re-uploading cached features

if RESUME_FEATURES:
    from google.colab import files
    print("Select X_features.npy and y_labels.npy together:")
    uploaded = files.upload()
    for fname in uploaded:
        dest = os.path.join(SSBD_DIR, fname)
        os.rename(fname, dest)
    X = np.load(os.path.join(SSBD_DIR, 'X_features.npy'))
    y = np.load(os.path.join(SSBD_DIR, 'y_labels.npy'))
    print("Loaded uploaded features. X shape:", X.shape, " y shape:", y.shape)

## Step 8 — Train / validation split and PyTorch DataLoaders

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

class SSBDDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

full_dataset = SSBDDataset(X, y)
val_size = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

print(f"Train samples: {train_size}, Val samples: {val_size}")

Train samples: 147, Val samples: 36


## Step 9 — Define the LSTM model

In [ ]:
class BehaviorLSTM(nn.Module):
    def __init__(self, input_size=FEATURES_PER_FRAME, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_step = out[:, -1, :]
        return self.fc(last_step)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BehaviorLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print("Model ready on device:", device)

Model ready on device: cpu


### Step 9b — Only run this if resuming training after a disconnect (ACTION NEEDED)

Skip on a first run. If you already have a downloaded `checkpoint.pth` from before, run this to re-upload it and continue from that epoch instead of starting over.

In [ ]:
RESUME_TRAINING = False  # change to True if you're re-uploading a checkpoint
start_epoch = 0

if RESUME_TRAINING:
    from google.colab import files
    print("Select your checkpoint.pth:")
    uploaded = files.upload()
    ckpt_name = list(uploaded.keys())[0]
    checkpoint = torch.load(ckpt_name, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch}")
else:
    print("Starting fresh from epoch 0")

## Step 10 — Train

Runs for 20 epochs total. Saves a checkpoint locally after **every** epoch, and **downloads it to your computer every 5 epochs** as your safeguard — matching what Anindya asked for, just without Drive. Your browser may ask to allow multiple downloads on this page; click **Allow**.

In [ ]:
start_epoch = 0


In [ ]:
from google.colab import files

CHECKPOINT_PATH = os.path.join(SSBD_DIR, 'checkpoint.pth')
TOTAL_EPOCHS = 20
DOWNLOAD_EVERY = 5

for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        correct += (preds.argmax(1) == yb).sum().item()
        total += xb.size(0)

    train_acc = correct / total
    train_loss /= total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_correct += (preds.argmax(1) == yb).sum().item()
            val_total += xb.size(0)
    val_acc = val_correct / max(val_total, 1)

    print(f"Epoch {epoch+1}/{TOTAL_EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.2%} val_acc={val_acc:.2%}")

    # Safeguard: save checkpoint locally every epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, CHECKPOINT_PATH)

    # Download to your computer every DOWNLOAD_EVERY epochs
    if (epoch + 1) % DOWNLOAD_EVERY == 0:
        files.download(CHECKPOINT_PATH)
        print(f"  -> Downloaded checkpoint after epoch {epoch+1}")

print("Training complete.")

Epoch 1/20 | train_loss=1.0460 train_acc=40.14% val_acc=47.22%
Epoch 2/20 | train_loss=0.8258 train_acc=66.67% val_acc=63.89%
Epoch 3/20 | train_loss=0.7791 train_acc=68.03% val_acc=55.56%
Epoch 4/20 | train_loss=0.7930 train_acc=63.95% val_acc=63.89%
Epoch 5/20 | train_loss=0.5698 train_acc=79.59% val_acc=77.78%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  -> Downloaded checkpoint after epoch 5
Epoch 6/20 | train_loss=0.5814 train_acc=75.51% val_acc=77.78%
Epoch 7/20 | train_loss=0.3547 train_acc=88.44% val_acc=80.56%
Epoch 8/20 | train_loss=0.2851 train_acc=89.80% val_acc=77.78%
Epoch 9/20 | train_loss=0.3209 train_acc=89.80% val_acc=77.78%
Epoch 10/20 | train_loss=0.2401 train_acc=91.16% val_acc=83.33%


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  -> Downloaded checkpoint after epoch 15
Epoch 16/20 | train_loss=0.0900 train_acc=95.92% val_acc=83.33%
Epoch 17/20 | train_loss=0.0945 train_acc=95.24% val_acc=83.33%
Epoch 18/20 | train_loss=0.0932 train_acc=94.56% val_acc=88.89%
Epoch 19/20 | train_loss=0.0693 train_acc=96.60% val_acc=88.89%
Epoch 20/20 | train_loss=0.0565 train_acc=96.60% val_acc=88.89%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  -> Downloaded checkpoint after epoch 20
Training complete.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

 armFlapping     1.0000    1.0000    1.0000         8
 headBanging     0.9231    0.8000    0.8571        15
    spinning     0.8000    0.9231    0.8571        13

    accuracy                         0.8889        36
   macro avg     0.9077    0.9077    0.9048        36
weighted avg     0.8957    0.8889    0.8889        36


Confusion Matrix:
[[ 8  0  0]
 [ 0 12  3]
 [ 0  1 12]]


In [ ]:
total_videos_found = sum(len(v) for v in class_videos.values())
sequences_produced = len(X)

print(f"Total videos found: {total_videos_found}")
print(f"Valid sequences produced: {sequences_produced}")
print(f"Success rate: {sequences_produced / total_videos_found * 100:.2f}%")

# How much of each sequence actually had a detected pose (non-zero) vs padded/missed frames
import numpy as np
frame_detected = (X.sum(axis=2) != 0)  # True where a frame has real landmark data
avg_frame_detection_rate = frame_detected.mean() * 100
print(f"Average frame-level pose detection rate: {avg_frame_detection_rate:.2f}%")

Total videos found: 183
Valid sequences produced: 183
Success rate: 100.00%
Average frame-level pose detection rate: 70.05%


## Step 11 — Save and download the final deliverable: `behavior_model.pth`
This is the exact file the group needs from you for Phase 3.

In [ ]:
FINAL_MODEL_PATH = os.path.join(SSBD_DIR, 'behavior_model.pth')
torch.save(model.state_dict(), FINAL_MODEL_PATH)

from google.colab import files
files.download(FINAL_MODEL_PATH)
print("Downloaded behavior_model.pth to your computer.")
print("Send this file to whoever is doing the fusion/deployment step.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded behavior_model.pth to your computer.
Send this file to whoever is doing the fusion/deployment step.


## Notes

- **If Colab disconnects partway through:** re-open this notebook, run Steps 1-6 again (feature extraction re-runs unless you use Step 7b to re-upload your cached features), then use Step 7b and/or Step 9b to re-upload your last downloaded backups instead of starting over.
- **If Step 5 finds 0 videos:** open the folder listing printed in Step 4, check the actual nested folder name, and adjust the `glob` pattern in Step 5.
- **Where your file ends up:** downloaded straight to your computer's Downloads folder as `behavior_model.pth` (and `checkpoint.pth` every 5 epochs along the way). Nothing depends on Google Drive in this version.